# Vacancies 2026 — Salary Prediction v2
**Target:** `salary_mean_net` | **Metric:** MAPE | **Models:** LightGBM + CatBoost ensemble

Key improvements over v1:
- `log1p` target transform (skew 1.53 → ~0)
- Target encoding for high-cardinality cols (inside CV, no leakage)
- Rich feature engineering: `has_skills`, `has_languages`, `skills_count`, city flags
- Separate TF-IDF on `name_clean` (job title signal)
- Dropped 92%-null branded description columns

### 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

SEED       = 42
N_SPLITS   = 5
SVD_DESC   = 80   # SVD components for lemmatized description
SVD_TITLE  = 30   # SVD components for job title
TFIDF_DESC  = 5000
TFIDF_TITLE = 2000
np.random.seed(SEED)

### 2. Data Loading

In [2]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test_x.csv')

print(f'train: {train.shape}, test: {test.shape}')

train: (49051, 26), test: (12263, 25)


### 3. Text Feature Engineering

In [3]:
DESC_COL  = 'lemmaized_wo_stopwords_raw_description'
TITLE_COL = 'name_clean'

for df in [train, test]:
    df[DESC_COL]  = df[DESC_COL].fillna('')
    df[TITLE_COL] = df[TITLE_COL].fillna('')

n_train = len(train)
corpus_desc  = pd.concat([train[DESC_COL],  test[DESC_COL]],  ignore_index=True)
corpus_title = pd.concat([train[TITLE_COL], test[TITLE_COL]], ignore_index=True)

# Description TF-IDF + SVD
tfidf_desc = TfidfVectorizer(max_features=TFIDF_DESC, sublinear_tf=True, min_df=3, ngram_range=(1, 2), dtype=np.float32)
svd_desc   = TruncatedSVD(n_components=SVD_DESC, random_state=SEED)
mat_desc   = svd_desc.fit_transform(tfidf_desc.fit_transform(corpus_desc)).astype(np.float32)

# Title TF-IDF + SVD (job title is a strong salary signal)
tfidf_title = TfidfVectorizer(max_features=TFIDF_TITLE, sublinear_tf=True, min_df=2, ngram_range=(1, 2), dtype=np.float32)
svd_title   = TruncatedSVD(n_components=SVD_TITLE, random_state=SEED)
mat_title   = svd_title.fit_transform(tfidf_title.fit_transform(corpus_title)).astype(np.float32)

desc_cols  = [f'desc_svd_{i}'  for i in range(SVD_DESC)]
title_cols = [f'title_svd_{i}' for i in range(SVD_TITLE)]

svd_train = pd.DataFrame(
    np.hstack([mat_desc[:n_train], mat_title[:n_train]]),
    columns=desc_cols + title_cols
)
svd_test = pd.DataFrame(
    np.hstack([mat_desc[n_train:], mat_title[n_train:]]),
    columns=desc_cols + title_cols
)

print(f'desc SVD explained var : {svd_desc.explained_variance_ratio_.sum():.3f}')
print(f'title SVD explained var: {svd_title.explained_variance_ratio_.sum():.3f}')

desc SVD explained var : 0.224
title SVD explained var: 0.256


### 4. Feature Engineering

In [4]:
EXPERIENCE_ORDER = ['Нет опыта', 'От 1 года до 3 лет', 'От 3 до 6 лет', 'Более 6 лет']

# Columns to drop before modelling
DROP_COLS = [
    'id', 'salary_mean_net',
    'raw_description',
    'raw_branded_description',                          # 92% null
    'lemmaized_wo_stopwords_raw_branded_description',   # 92% null
    'lemmaized_wo_stopwords_raw_description',
    'name',        # superseded by name_clean
    'name_clean',  # encoded via TF-IDF SVD
]

LOW_CARD_COLS = [
    'schedule_name', 'employment_name',
    'is_branded_description', 'if_foreign_language',
    'accept_handicapped', 'accept_kids',
]

# High-card cols for target encoding (inside CV)
TE_COLS = [
    'employer_id', 'employer_name',
    'professional_roles_name', 'specializations_profarea_name',
    'employer_industries',
    'unified_address_city', 'unified_address_state', 'unified_address_region',
]


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    exp_map = {v: i for i, v in enumerate(EXPERIENCE_ORDER)}
    df['experience_ord'] = df['experience_name'].map(exp_map).fillna(-1).astype(np.int8)

    df['desc_word_count'] = df[DESC_COL].str.split().str.len().fillna(0).astype(np.int16)

    # skills / languages flags
    df['has_skills']    = (df['key_skills_name'].fillna('[]') != '[]').astype(np.int8)
    df['has_languages'] = (df['languages_name'].fillna('[]') != '[]').astype(np.int8)
    # rough skills count from string length heuristic
    df['skills_count']  = df['key_skills_name'].fillna('[]').str.count(',').astype(np.int16)

    # city-level flags (strong salary signal from EDA)
    city = df['unified_address_city'].fillna('').str.lower()
    df['is_moscow'] = city.str.contains('москва').astype(np.int8)
    df['is_spb']    = city.str.contains('санкт').astype(np.int8)

    # fill NaN in TE cols with placeholder before encoding
    for col in TE_COLS:
        if col in df.columns:
            df[col] = df[col].fillna('__NA__').astype(str)

    return df


train = engineer_features(train)
test  = engineer_features(test)

# OrdinalEncoder for low-card cols
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
train[LOW_CARD_COLS] = oe.fit_transform(train[LOW_CARD_COLS].astype(str))
test[LOW_CARD_COLS]  = oe.transform(test[LOW_CARD_COLS].astype(str))

TARGET = 'salary_mean_net'
y_raw  = train[TARGET].values.astype(np.float64)
y      = np.log1p(y_raw).astype(np.float32)  # log1p transform: skew 1.53 → ~0

# unified_address_country has 1 unique value → useless; drop all remaining object cols not yet handled
EXTRA_DROP = ['unified_address_country']

base_feature_cols = [
    c for c in train.columns
    if c not in DROP_COLS + TE_COLS + EXTRA_DROP + ['experience_name', 'key_skills_name', 'languages_name']
    and train[c].dtype != object
]

X_base_train = train[base_feature_cols].reset_index(drop=True)
X_base_test  = test[[c for c in base_feature_cols if c in test.columns]].reset_index(drop=True)
X_base_test  = X_base_test.reindex(columns=X_base_train.columns)

print(f'Base features: {X_base_train.shape[1]}')
print(f'TE cols to encode: {TE_COLS}')

Base features: 13
TE cols to encode: ['employer_id', 'employer_name', 'professional_roles_name', 'specializations_profarea_name', 'employer_industries', 'unified_address_city', 'unified_address_state', 'unified_address_region']


### 5. LightGBM — K-Fold with In-Fold Target Encoding

In [5]:
lgb_params = {
    'objective'       : 'regression',   # MSE on log1p target → equivalent to optimising log-MAPE
    'metric'          : 'rmse',
    'n_estimators'    : 5000,
    'learning_rate'   : 0.02,
    'num_leaves'      : 127,
    'max_depth'       : -1,
    'min_child_samples': 20,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq'    : 5,
    'reg_alpha'       : 0.1,
    'reg_lambda'      : 0.1,
    'random_state'    : SEED,
    'n_jobs'          : -1,
    'verbose'         : -1,
}

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(train), dtype=np.float64)
pred_lgb = np.zeros(len(test),  dtype=np.float64)

global_te_mean = np.mean(y)  # fallback for unseen categories

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_base_train)):
    # --- in-fold target encoding (no leakage) ---
    te_frames_tr  = []
    te_frames_val = []
    te_frames_te  = []

    for col in TE_COLS:
        col_tr  = train[col].iloc[tr_idx].reset_index(drop=True)
        col_val = train[col].iloc[val_idx].reset_index(drop=True)
        col_te  = test[col].reset_index(drop=True)

        te_map = pd.Series(y[tr_idx]).groupby(col_tr).mean()

        te_frames_tr.append(col_tr.map(te_map).fillna(global_te_mean).rename(f'te_{col}'))
        te_frames_val.append(col_val.map(te_map).fillna(global_te_mean).rename(f'te_{col}'))
        te_frames_te.append(col_te.map(te_map).fillna(global_te_mean).rename(f'te_{col}'))

    te_tr  = pd.concat(te_frames_tr,  axis=1)
    te_val = pd.concat(te_frames_val, axis=1)
    te_te  = pd.concat(te_frames_te,  axis=1)

    X_tr  = pd.concat([X_base_train.iloc[tr_idx].reset_index(drop=True),  svd_train.iloc[tr_idx].reset_index(drop=True),  te_tr],  axis=1)
    X_val = pd.concat([X_base_train.iloc[val_idx].reset_index(drop=True), svd_train.iloc[val_idx].reset_index(drop=True), te_val], axis=1)
    X_te  = pd.concat([X_base_test.reset_index(drop=True),                svd_test.reset_index(drop=True),                te_te],  axis=1)

    y_tr, y_val_raw = y[tr_idx], y_raw[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y[tr_idx],
        eval_set=[(X_val, y[val_idx])],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(1000),
        ],
    )

    val_preds_raw = np.expm1(model.predict(X_val))
    oof_lgb[val_idx] = val_preds_raw
    pred_lgb        += np.expm1(model.predict(X_te)) / N_SPLITS

    fold_mape = np.mean(np.abs((y_val_raw - val_preds_raw) / (y_val_raw + 1e-8)))
    print(f'Fold {fold+1} | LGB MAPE: {fold_mape:.4f} | best_iter: {model.best_iteration_}')

lgb_oof_mape = np.mean(np.abs((y_raw - oof_lgb) / (y_raw + 1e-8)))
print(f'\nLightGBM OOF MAPE: {lgb_oof_mape:.4f}')

[1000]	valid_0's rmse: 0.36449
[2000]	valid_0's rmse: 0.362338
[3000]	valid_0's rmse: 0.361807
Fold 1 | LGB MAPE: 0.2663 | best_iter: 3366
[1000]	valid_0's rmse: 0.349152
[2000]	valid_0's rmse: 0.346578
[3000]	valid_0's rmse: 0.345838
[4000]	valid_0's rmse: 0.345574
Fold 2 | LGB MAPE: 0.2527 | best_iter: 4596
[1000]	valid_0's rmse: 0.362103
[2000]	valid_0's rmse: 0.36011
[3000]	valid_0's rmse: 0.359566
[4000]	valid_0's rmse: 0.359416
Fold 3 | LGB MAPE: 0.2617 | best_iter: 4201
[1000]	valid_0's rmse: 0.354401
[2000]	valid_0's rmse: 0.352379
[3000]	valid_0's rmse: 0.35193
[4000]	valid_0's rmse: 0.351771
Fold 4 | LGB MAPE: 0.2595 | best_iter: 4308
[1000]	valid_0's rmse: 0.350836
[2000]	valid_0's rmse: 0.348685
[3000]	valid_0's rmse: 0.348225
Fold 5 | LGB MAPE: 0.2582 | best_iter: 3642

LightGBM OOF MAPE: 0.2597


### 6. CatBoost — K-Fold with Native Categorical + In-Fold TE

In [6]:
cb_params = dict(
    loss_function    = 'RMSE',   # on log1p target
    eval_metric      = 'RMSE',
    iterations       = 5000,
    learning_rate    = 0.02,
    depth            = 7,
    l2_leaf_reg      = 3.0,
    random_strength  = 1.0,
    bagging_temperature = 0.5,
    od_type          = 'Iter',
    od_wait          = 150,
    random_seed      = SEED,
    thread_count     = -1,
    verbose          = 1000,
)

oof_cb  = np.zeros(len(train), dtype=np.float64)
pred_cb = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_base_train)):
    # same in-fold TE as LGB
    te_frames_tr  = []
    te_frames_val = []
    te_frames_te  = []

    for col in TE_COLS:
        col_tr  = train[col].iloc[tr_idx].reset_index(drop=True)
        col_val = train[col].iloc[val_idx].reset_index(drop=True)
        col_te  = test[col].reset_index(drop=True)

        te_map = pd.Series(y[tr_idx]).groupby(col_tr).mean()

        te_frames_tr.append(col_tr.map(te_map).fillna(global_te_mean).rename(f'te_{col}'))
        te_frames_val.append(col_val.map(te_map).fillna(global_te_mean).rename(f'te_{col}'))
        te_frames_te.append(col_te.map(te_map).fillna(global_te_mean).rename(f'te_{col}'))

    te_tr  = pd.concat(te_frames_tr,  axis=1)
    te_val = pd.concat(te_frames_val, axis=1)
    te_te  = pd.concat(te_frames_te,  axis=1)

    X_tr  = pd.concat([X_base_train.iloc[tr_idx].reset_index(drop=True),  svd_train.iloc[tr_idx].reset_index(drop=True),  te_tr],  axis=1)
    X_val = pd.concat([X_base_train.iloc[val_idx].reset_index(drop=True), svd_train.iloc[val_idx].reset_index(drop=True), te_val], axis=1)
    X_te  = pd.concat([X_base_test.reset_index(drop=True),                svd_test.reset_index(drop=True),                te_te],  axis=1)

    # CatBoost doesn't need category dtype here — all TE cols are already numeric
    train_pool = Pool(X_tr,  y[tr_idx])
    val_pool   = Pool(X_val, y[val_idx])

    model = CatBoostRegressor(**cb_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_preds_raw    = np.expm1(model.predict(val_pool))
    oof_cb[val_idx]  = val_preds_raw
    pred_cb         += np.expm1(model.predict(Pool(X_te))) / N_SPLITS

    fold_mape = np.mean(np.abs((y_raw[val_idx] - val_preds_raw) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | CB MAPE: {fold_mape:.4f}')

cb_oof_mape = np.mean(np.abs((y_raw - oof_cb) / (y_raw + 1e-8)))
print(f'\nCatBoost OOF MAPE: {cb_oof_mape:.4f}')

0:	learn: 0.5271318	test: 0.5291437	best: 0.5291437 (0)	total: 64.6ms	remaining: 5m 22s
1000:	learn: 0.1902792	test: 0.3785273	best: 0.3785273 (1000)	total: 7.32s	remaining: 29.2s
2000:	learn: 0.1694278	test: 0.3704796	best: 0.3704796 (2000)	total: 14.5s	remaining: 21.8s
3000:	learn: 0.1540978	test: 0.3664489	best: 0.3664489 (3000)	total: 21.7s	remaining: 14.5s
4000:	learn: 0.1413772	test: 0.3643248	best: 0.3643192 (3999)	total: 29s	remaining: 7.24s
4999:	learn: 0.1303125	test: 0.3628212	best: 0.3628212 (4999)	total: 36.7s	remaining: 0us

bestTest = 0.362821223
bestIteration = 4999

Fold 1 | CB MAPE: 0.2770
0:	learn: 0.5263357	test: 0.5323294	best: 0.5323294 (0)	total: 9.42ms	remaining: 47.1s
1000:	learn: 0.1913239	test: 0.3676080	best: 0.3676080 (1000)	total: 7.78s	remaining: 31.1s
2000:	learn: 0.1708477	test: 0.3591046	best: 0.3591046 (2000)	total: 15.5s	remaining: 23.3s
3000:	learn: 0.1555707	test: 0.3549985	best: 0.3549985 (3000)	total: 23.1s	remaining: 15.4s
4000:	learn: 0.1430590

### 7. Ensemble Blending (OOF-weighted)

In [7]:
w_lgb = 1.0 / lgb_oof_mape
w_cb  = 1.0 / cb_oof_mape
w_sum = w_lgb + w_cb

oof_blend  = (w_lgb * oof_lgb  + w_cb * oof_cb)  / w_sum
pred_blend = (w_lgb * pred_lgb + w_cb * pred_cb) / w_sum

blend_mape = np.mean(np.abs((y_raw - oof_blend) / (y_raw + 1e-8)))
print(f'LGB weight : {w_lgb/w_sum:.3f}')
print(f'CB weight  : {w_cb/w_sum:.3f}')
print(f'Ensemble OOF MAPE: {blend_mape:.4f}')

LGB weight : 0.510
CB weight  : 0.490
Ensemble OOF MAPE: 0.2640


### 8. Post-processing & Submission

In [8]:
final_preds = np.clip(pred_blend, a_min=0, a_max=None)

test_ids = pd.read_csv('data/test_x.csv', usecols=['id'])['id']

submission = pd.DataFrame({
    'id': test_ids.values,
    'salary_mean_net': final_preds,
})

submission.to_csv('submission_v2.csv', index=False)
print(submission.head(10))
print(f'\nSubmission shape : {submission.shape}')
print(f'Negative preds   : {(final_preds < 0).sum()}')
print(f'min={final_preds.min():.0f}  median={np.median(final_preds):.0f}  max={final_preds.max():.0f}')

         id  salary_mean_net
0  46224201    120698.599044
1  42119402     44184.509868
2  45716401     28844.689924
3  43716203     33879.112819
4  47109602     54967.402106
5  45507200     34425.138656
6  41123802     61267.670239
7  49155602     40525.021588
8  41169603     23002.391154
9  48041400     55051.788397

Submission shape : (12263, 2)
Negative preds   : 0
min=11573  median=43769  max=163333
